# Week 13 Student Lab Scaffold — LLM and RAG in Healthcare

This is the **student challenge version**. You will not receive the full implementation first.

Your job is to translate pseudocode into working code. If you get stuck, use the hint cells. The instructor/reference notebook contains the complete solution.

> Clinical safety note: this notebook is for education only. Do not use generated output for patient care.


## 0. Setup

Run this cell first. It installs dependencies, loads common libraries, and configures Gemini if `GOOGLE_API_KEY` is available. If no API key is found, the lab uses fallback responses.


In [14]:
# ============================================================
# Section 0: Environment Setup
# ============================================================

!pip install langchain langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers google-generativeai --quiet

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

USE_GEMINI = True
llm = None

try:
    import google.generativeai as genai
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception:
        API_KEY = os.environ.get("GOOGLE_API_KEY", "")

    if API_KEY:
        genai.configure(api_key=API_KEY)
        llm = genai.GenerativeModel("gemini-1.5-flash")
        USE_GEMINI = True
        print("Gemini API configured.")
    else:
        print("No API key found. Using fallback mode.")
except Exception as e:
    print(f"Gemini unavailable: {e}. Using fallback mode.")

print(f"Runtime mode: {'Gemini' if USE_GEMINI else 'Fallback'}")


No API key found. Using fallback mode.
Runtime mode: Gemini


In [23]:
!pip install transformers accelerate -q

from transformers import pipeline

USE_QWEN = True

qwen_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## 1. Helper Function

This helper lets the notebook run even without a live LLM API.


In [24]:
def _fallback_response(prompt):
    prompt_lower = prompt.lower()
    if "insufficient" in prompt_lower or "not covered" in prompt_lower:
        return "Insufficient evidence in the retrieved documents. The available context does not support a safe clinical answer."
    if "reference" in prompt_lower or "cite" in prompt_lower:
        return (
            "Example response for teaching: Ceftriaxone is commonly used for suspected meningococcemia. "
            "However, any cited references must be manually verified. This fallback intentionally does not provide real citations."
        )
    if "fever" in prompt_lower and "rash" in prompt_lower:
        return (
            "Structured fallback answer:\n"
            "Key findings: fever, petechial rash, hypotension, mild neck stiffness.\n"
            "Top urgent differential: meningococcemia / meningitis with sepsis.\n"
            "Immediate actions: sepsis protocol, blood cultures, empiric antibiotics, urgent senior review.\n"
            "Missing information: exposure history, immunization status, labs, lactate, platelets, coagulation profile."
        )
    return "Fallback response: build a structured answer and state uncertainty when evidence is missing."

'''
def query_llm(prompt, max_tokens=1024):

    if USE_GEMINI and llm is not None:
        try:
            response = llm.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(max_output_tokens=max_tokens, temperature=0.3),
            )
            return response.text
        except Exception as e:
            print(f"Gemini error: {e}. Using fallback.")
            return _fallback_response(prompt)
    return _fallback_response(prompt)
'''

def query_llm(prompt, max_tokens=1024):
    if USE_QWEN and qwen_pipe is not None:
        try:
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a careful assistant. "
                        "Answer based only on the provided context. "
                        "If evidence is insufficient, say so clearly."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]

            response = qwen_pipe(
                messages,
                max_new_tokens=max_tokens,
                do_sample=True,
                temperature=0.3,
                return_full_text=False,
            )

            return response[0]["generated_text"]

        except Exception as e:
            print(f"Qwen error: {e}. Using fallback.")
            return _fallback_response(prompt)

    return _fallback_response(prompt)

## 2. The 3 AM Clinical Case

Read the case. Before writing code, write your own top 3 differentials in a text cell or on paper.


In [25]:

clinical_case = """
Patient: 34-year-old male
Presenting at: 3 AM, Emergency Department
Chief complaint: Fever, abdominal pain, rash

Vitals:
- Temperature: 38.9 C
- Heart rate: 112 bpm
- Blood pressure: 95/60 mmHg
- Respiratory rate: 22/min
- SpO2: 96% on room air

Physical exam:
- Diffuse, non-blanching petechial rash on trunk and extremities
- Diffuse abdominal tenderness, no rebound
- Mild neck stiffness
- Alert but appears toxic

History:
- No recent travel
- No known drug allergies
- No significant past medical history
- No recent antibiotics
"""
print(clinical_case)



Patient: 34-year-old male
Presenting at: 3 AM, Emergency Department
Chief complaint: Fever, abdominal pain, rash

Vitals:
- Temperature: 38.9 C
- Heart rate: 112 bpm
- Blood pressure: 95/60 mmHg
- Respiratory rate: 22/min
- SpO2: 96% on room air

Physical exam:
- Diffuse, non-blanching petechial rash on trunk and extremities
- Diffuse abdominal tenderness, no rebound
- Mild neck stiffness
- Alert but appears toxic

History:
- No recent travel
- No known drug allergies
- No significant past medical history
- No recent antibiotics



# Lab 1 — Direct LLM + Structured Clinical Prompting

## Challenge 1A: Build the prompt yourself

Pseudocode:

```text
DEFINE clinical_case
BUILD a structured clinical prompt asking for:
  - key findings
  - top 5 differential diagnoses ranked by urgency
  - immediate actions
  - key labs
  - uncertainty / missing information
CALL query_llm(prompt)
PRINT response
```

Do not look at the reference notebook yet.


In [26]:
# TODO: Build your structured clinical prompt.
# Requirement: include key findings, differential, immediate actions, labs, and uncertainty.

prompt_structured = f"""
Please provide a structured answer to the following clinical case, and it is required to contains key findings, differential, immediate actions, labs, and uncertainty.
Please  make the answer like structrue:
  - key findings
  - top 5 differential diagnoses ranked by urgency
  - immediate actions
  - key labs
  - uncertainty / missing information

Clinical case:
{clinical_case}



"""
print(prompt_structured)

response_structured = query_llm(prompt_structured)
print(response_structured)


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Please provide a structured answer to the following clinical case, and it is required to contains key findings, differential, immediate actions, labs, and uncertainty.
Please  make the answer like structrue:
  - key findings
  - top 5 differential diagnoses ranked by urgency
  - immediate actions
  - key labs
  - uncertainty / missing information

Clinical case:

Patient: 34-year-old male
Presenting at: 3 AM, Emergency Department
Chief complaint: Fever, abdominal pain, rash

Vitals:
- Temperature: 38.9 C
- Heart rate: 112 bpm
- Blood pressure: 95/60 mmHg
- Respiratory rate: 22/min
- SpO2: 96% on room air

Physical exam:
- Diffuse, non-blanching petechial rash on trunk and extremities
- Diffuse abdominal tenderness, no rebound
- Mild neck stiffness
- Alert but appears toxic

History:
- No recent travel
- No known drug allergies
- No significant past medical history
- No recent antibiotics





### Structured Answer

#### Key Findings
- **Fever:** 38.9°C (102°F)
- **Abdominal Pain:** Pe

### Hint 1 — Response Schema

Ask the model to use this format:

```text
1. Key findings
2. Most urgent diagnoses, ranked
3. Immediate actions in the first 10 minutes
4. Labs and tests
5. Missing information / uncertainty
6. Safety warning
```


## Challenge 1B: Citation Stress Test

Pseudocode:

```text
BUILD a prompt asking for treatment recommendation + specific citations
FOR each citation in response:
  CHECK if citation exists
  CHECK if citation supports the exact claim
CREATE a verification table
REWRITE the answer in a safer form
```


In [27]:
# TODO: Ask for references, then verify them manually.
# Your output should include a verification table with claim / citation / exists? / supports claim?

prompt_citation_test = f"""
FOR each citation in response:
  CHECK if citation exists
  CHECK if citation supports the exact claim
CREATE a verification table
REWRITE the answer in a safer form

Clinical case:
{clinical_case}
"""

response_citation = query_llm(prompt_citation_test, max_tokens=1500)
print(response_citation)

# TODO: Fill this table after manual verification.
verification_table = pd.DataFrame([
    {"claim": "", "citation": "", "exists": "", "supports_claim": "", "notes": ""},
])
verification_table


Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```markdown
### Verification Table

| Citation | Claim | Evidence | Support |
|---------|-------|----------|--------|
| Patient's Chief Complaint | Fever, abdominal pain, rash | Presenting at 3 AM, Emergency Department, Chief complaint of fever, abdominal pain, and rash | Evidence of patient's symptoms |
| Vital Signs | Temperature: 38.9 C | Patient's temperature is within normal range (36.1°C - 37.2°C) | Normal body temperature |
| Vitals | Heart Rate: 112 bpm | Patient has a heart rate of 112 beats per minute | High heart rate indicates potential for infection or other complications |
| Blood Pressure | Blood Pressure: 95/60 mmHg | Patient's blood pressure is within normal range (120/80 mmHg) | Normal blood pressure |
| Respiratory Rate | Respiratory Rate: 22/min | Patient has a respiratory rate of 22 breaths per minute | Low respiratory rate suggests possible respiratory distress |
| SpO2 | SpO2: 96% on room air | Patient's SpO2 level is 96%, indicating adequate oxygenation | Normal

,claim,citation,exists,supports_claim,notes
0,,,,,


# Lab 2 — Build a Clinical RAG System

You will now build the system that prevents the model from relying only on memory.

## Challenge 2A: Chunk the guideline corpus

Pseudocode:

```text
LOAD guideline_texts
JOIN into all_text
SPLIT all_text into overlapping chunks
PRINT number of chunks
PLOT chunk length distribution
```


In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

clinical_guidelines = [
    """SEPSIS MANAGEMENT SUMMARY
    Sepsis is life-threatening organ dysfunction caused by dysregulated host response to infection.
    Obtain blood cultures before antibiotics when feasible. Measure serum lactate.
    Administer broad-spectrum IV antibiotics rapidly after sepsis recognition.
    Fluid resuscitation and vasopressors may be needed for shock.
    """,
    """MENINGOCOCCEMIA SUMMARY
    Suspected meningococcemia is a medical emergency. Fever, petechial rash, hypotension,
    and toxic appearance should trigger immediate evaluation and empiric therapy.
    Ceftriaxone or cefotaxime are commonly used empiric options, adjusted for local guidance.
    Public health notification and prophylaxis for close contacts may be required.
    """,
    """BACTERIAL MENINGITIS SUMMARY
    Bacterial meningitis may present with fever, neck stiffness, altered mental status, and sepsis.
    Blood cultures should be obtained promptly. Do not delay empiric antibiotics for imaging if unstable.
    Dexamethasone may be considered before or with the first antibiotic dose depending on context.
    """,
    """HEALTH DATA PRIVACY SUMMARY
    Clinical AI systems processing PHI require privacy safeguards. Common controls include encryption,
    access control, audit logs, de-identification when appropriate, and contractual safeguards for vendors.
    """,
]

# TODO: import the splitter, join the text, split into chunks, and inspect chunk lengths.

# from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_text = "\n\n".join(clinical_guidelines)
chunks = text_splitter.split_text(all_text)
print(f"\nChunking Results:")
print(f"  Chunk size: 500 characters")
print(f"  Chunk overlap: 100 characters")
print(f"  Total chunks: {len(chunks)}")
print(f"  Avg chunk length: {np.mean([len(c) for c in chunks]):.0f} chars")
print(f"  Min chunk length: {min(len(c) for c in chunks)} chars")
print(f"  Max chunk length: {max(len(c) for c in chunks)} chars")

print(f"\n--- Sample Chunk (first) ---")
print(chunks[0][:300] + "...")




# TODO: create splitter
# splitter = ...
# chunks = ...

# print(f"Total chunks: {len(chunks)}")
# print(chunks[0][:300])



Chunking Results:
  Chunk size: 500 characters
  Chunk overlap: 100 characters
  Total chunks: 4
  Avg chunk length: 324 chars
  Min chunk length: 238 chars
  Max chunk length: 372 chars

--- Sample Chunk (first) ---
SEPSIS MANAGEMENT SUMMARY
    Sepsis is life-threatening organ dysfunction caused by dysregulated host response to infection.
    Obtain blood cultures before antibiotics when feasible. Measure serum lactate.
    Administer broad-spectrum IV antibiotics rapidly after sepsis recognition.
    Fluid re...


### Hint 2A — Function Names

You probably want:

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)
chunks = splitter.split_text(all_text)
```


## Challenge 2B: Build embeddings and vector store

Pseudocode:

```text
WRAP chunks as Document objects
LOAD embedding model
BUILD Chroma vector store
CREATE retriever with k=5
TEST one similarity search
```


In [31]:
# TODO: Build the vector store.

# from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_community.vectorstores import Chroma
# from langchain_core.documents import Document

# documents = ...
# embeddings = ...
# vectorstore = ...
# retriever = ...

# test_query = "What should we do for suspected meningococcemia?"
# retrieved_docs = retriever.invoke(test_query)
# print(retrieved_docs[0].page_content[:500])


from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

documents = [
    Document(page_content=chunk, metadata={"chunk_id": i, "source": "clinical_guidelines"})
    for i, chunk in enumerate(chunks)
]

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="student_clinical_rag"
)

test_embedding = embeddings.embed_query("MENINGOCOCCEMIA guideline")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"Sample values: [{test_embedding[0]:.4f}, {test_embedding[1]:.4f}, ...]")


print(f"Vector store created!")
print(f"  Collection: clinical_guidelines")
print(f"  Documents stored: {vectorstore._collection.count()}")
print(f"  Embedding model: all-MiniLM-L6-v2 ({len(test_embedding)}-dim)")

# --- Test similarity search ---
print("\n--- Test Retrieval ---")
test_query = "What antibiotic for meningococcemia?"
test_results = vectorstore.similarity_search_with_relevance_scores(test_query, k=3)

for i, (doc, score) in enumerate(test_results):
    print(f"\nResult {i+1} (relevance: {score:.4f}):")
    print(f"  {doc.page_content[:150]}...")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384
Sample values: [0.0071, -0.0284, ...]
Vector store created!
  Collection: clinical_guidelines
  Documents stored: 12
  Embedding model: all-MiniLM-L6-v2 (384-dim)

--- Test Retrieval ---

Result 1 (relevance: 0.4308):
  MENINGOCOCCEMIA SUMMARY
    Suspected meningococcemia is a medical emergency. Fever, petechial rash, hypotension,
    and toxic appearance should trig...

Result 2 (relevance: 0.4308):
  MENINGOCOCCEMIA SUMMARY
    Suspected meningococcemia is a medical emergency. Fever, petechial rash, hypotension,
    and toxic appearance should trig...

Result 3 (relevance: 0.4308):
  MENINGOCOCCEMIA SUMMARY
    Suspected meningococcemia is a medical emergency. Fever, petechial rash, hypotension,
    and toxic appearance should trig...


### Hint 2B — Object Constructors

```python
documents = [Document(page_content=c, metadata={"chunk_id": i}) for i, c in enumerate(chunks)]
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name="student_clinical_rag")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
```


## Challenge 2C: Write the RAG functions

Pseudocode:

```text
FUNCTION retrieve_context(question):
  retrieved_docs = retriever.invoke(question)
  context = join docs as [Passage 1], [Passage 2]...
  return context, retrieved_docs

FUNCTION rag_answer(question):
  context, docs = retrieve_context(question)
  prompt = instructions + context + question
  answer = query_llm(prompt)
  return answer, docs
```


In [32]:


retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

RAG_TEMPLATE = """You are a clinical AI assistant. Given the patient's clinical case and the provided context passages (clinical guidelines), answer the clinical question. If the context does not contain enough information to answer, say "Insufficient evidence in the retrieved documents."

PATIENT CLINICAL CASE:
{clinical_case}

CONTEXT (CLINICAL GUIDELINES):
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Base your answer ONLY on the provided CONTEXT (CLINICAL GUIDELINES).
2. Use the PATIENT CLINICAL CASE to apply the guidelines and formulate a relevant answer to the question.
3. Cite which passage from CONTEXT (CLINICAL GUIDELINES) supports each recommendation.
4. If information is missing from the CONTEXT (CLINICAL GUIDELINES), explicitly state what is missing.
5. Prioritize patient safety in all recommendations.

ANSWER:"""


def retrieve_context(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n---\n\n".join([
        f"[Passage {i+1}] {doc.page_content}"
        for i, doc in enumerate(retrieved_docs)
    ])
    return context, retrieved_docs
    raise NotImplementedError("Implement retrieve_context")


def rag_answer(question):
    context, docs = retrieve_context(question)
    prompt = RAG_TEMPLATE.format(clinical_case=clinical_case, context=context, question=question)
    answer = query_llm(prompt, max_tokens=1500)
    return {"answer": answer, "docs": docs, "context": context}
    raise NotImplementedError("Implement rag_answer")

print("=" * 60)
print("RAG QUERY -- Grounded in Retrieved Evidence")
print("=" * 60)

test_question = "What is the recommended empiric antibiotic for suspected meningococcemia in a 34-year-old adult?"
print(test_question)
result = rag_answer(test_question)

print(result["answer"])
print(f"\nRetrieved Passages (previews):")
for i, doc in enumerate(result["docs"]):
    print(f"  [{i+1}] {doc.page_content[:100]}...")

Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG QUERY -- Grounded in Retrieved Evidence
What is the recommended empiric antibiotic for suspected meningococcemia in a 34-year-old adult?
Based on the given CONTEXT (CLINICAL GUIDELINES), the recommended empiric antibiotic for suspected meningococcemia in a 34-year-old adult is:

Ceftriaxone or cefotaxime

The passage states that "Ceftriaxone or cefotaxime are commonly used empiric options, adjusted for local guidance." This directly supports this recommendation.

However, there is no specific mention of blood cultures being taken in the CONTEXT (CLINICAL GUIDELINES). Therefore, we cannot determine whether dexamethasone might be considered before or with the first antibiotic dose based solely on the information provided.

In conclusion, the most appropriate antibiotic recommendation would be Ceftriaxone or cefotaxime, as supported by the CONTEXT (CLINICAL GUIDELINES) and the lack of specific blood culture data.

Retrieved Passages (previews):
  [1] MENINGOCOCCEMIA SUMMARY
    Suspec

## Challenge 2D: Test covered vs not-covered questions

Run one question that should be covered by the corpus and one question that is not covered.

Your RAG system should answer the covered question and say insufficient evidence for the not-covered question.


In [33]:
# TODO: Test your RAG system.

covered_question = "What immediate actions are recommended for suspected meningococcemia?"
not_covered_question = "What is the recommended HbA1c target for elderly diabetes patients with multiple comorbidities?"

covered_result = rag_answer(covered_question)

print("=== Covered question ===")
print("Question:", covered_question)
print("\nAnswer:")
print(covered_result["answer"])
print("\nRetrieved context preview:")
print(covered_result["context"][:800])


not_covered_result = rag_answer(not_covered_question)

print("\n\n=== Not-covered question ===")
print("Question:", not_covered_question)
print("\nAnswer:")
print(not_covered_result["answer"])
print("\nRetrieved context preview:")
print(not_covered_result["context"][:800])

Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Covered question ===
Question: What immediate actions are recommended for suspected meningococcemia?

Answer:
Based on the provided CONTEXT (CLINICAL GUIDELINES), the following immediate actions are recommended for suspected meningococcemia:

1. Immediate evaluation of the patient by a healthcare provider.
2. Empiric use of empirical antibiotics, adjusting dosages as needed.
3. Prompt collection of blood cultures to confirm the diagnosis.
4. Administration of dexamethasone, especially if there is concern about brain involvement or instability.
5. Implementation of public health notification and prophylactic measures for close contacts.

These actions are supported by the relevant passages from the Clinical Guidelines, particularly Passage 1, which emphasizes the importance of prompt evaluation and empiric treatment for suspected meningococcemia. The other passages do not provide specific recommendations for immediate actions beyond those outlined in Passage 1.

Retrieved context pr

# Optional Mini-Lab — MedGemma

MedGemma is Google’s open medical model family built on Gemma. In this mini-lab, you first define the safety boundary, then optionally run inference if your environment supports it.

## Challenge: From model card to safe prototype

Pseudocode:

```text
READ the MedGemma model card
IDENTIFY:
  - model variant
  - input modality
  - output modality
  - intended use boundary
  - required validation

IF GPU + Hugging Face access are available:
  LOAD google/medgemma-1.5-4b-it
  RUN one non-PHI medical image/text example
  LABEL output as preliminary description, not diagnosis
ELSE:
  WRITE deployment pseudocode and validation plan
```


In [10]:


# TODO: Fill this table after reading the MedGemma model card.
medgemma_review = pd.DataFrame([
    {"question": "Which MedGemma variant would you use?", "your_answer": ""},
    {"question": "What input modality does your use case need?", "your_answer": ""},
    {"question": "What is the output allowed to claim?", "your_answer": ""},
    {"question": "What must be validated before clinical deployment?", "your_answer": ""},
    {"question": "Where does RAG still help?", "your_answer": ""},
])
medgemma_review





,question,your_answer
0,Which MedGemma variant would you use?,
1,What input modality does your use case need?,
2,What is the output allowed to claim?,
3,What must be validated before clinical deploym...,
4,Where does RAG still help?,


### Optional inference scaffold

Keep `RUN_MEDGEMMA = False` unless your instructor confirms that GPU and model access are available. Some MedGemma models may require accepting terms on Hugging Face.


In [11]:
RUN_MEDGEMMA = False

if RUN_MEDGEMMA:
    # TODO: Install and import the needed libraries.
    # !pip install transformers accelerate pillow --quiet
    # from transformers import pipeline
    # import torch

    # TODO: Load the model. You may need Hugging Face access approval.
    # pipe = pipeline(
    #     "image-text-to-text",
    #     model="google/medgemma-1.5-4b-it",
    #     torch_dtype=torch.bfloat16,
    #     device_map="auto",
    # )

    # TODO: Use only public, non-PHI images or text.
    # messages = [{"role": "user", "content": [
    #     {"type": "text", "text": "Describe the visible medical findings. Do not provide a final diagnosis."},
    #     {"type": "image", "url": "YOUR_PUBLIC_IMAGE_URL"},
    # ]}]
    # output = pipe(text=messages, max_new_tokens=256)
    # print(output)
    pass
else:
    print("MedGemma inference skipped. Complete the model-card review and deployment pseudocode instead.")


MedGemma inference skipped. Complete the model-card review and deployment pseudocode instead.


## Final Reflection

Answer these in prose:

1. Which part of RAG was easiest to build?
2. Which part is most likely to fail clinically?
3. What changed when you inspected retrieved passages before reading the generated answer?
4. What governance would a hospital need before using this with PHI?
